<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## See Which Clock Changes Are Allowed

**Understand constrained improvement by changing the clock settings together.**

Keep the same four observations, squared-error score, and exact calibration. The geometry comes from a simple fact: increasing one setting requires decreasing the other to remain calibrated.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Keep the equality while changing settings

The recorded errors are $-2,-1,-1,0$ min at $0,1,2,3$ h. The decision is correction $q$ in min and rate $r$ in min/h. The remaining errors are $y_i=d_i+q+rt_i$, for $i=1,\ldots,4$, and $\phi(q,r)=\sum_{i=1}^{4}y_i^2$.

Calibration at one hour requires $q+(1\,\mathrm h)r=1\,\mathrm{min}$. 

In numerical coordinates, write $h=q+r-1=0$. Allowed changes therefore satisfy

$$
\Delta q+\Delta r=0.
$$

The symbol $\Delta$ means a change. For example, adding 0.1 to $q$ and subtracting 0.1 from $r$ preserves the equality. The vector $(1,-1)$ points along the feasible line: it is a **tangent direction**. The vector $(1,1)$ points across it: it is a **normal direction**.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Compare an allowed improvement with a rejected one

Start at $(q,r)=(1,0)$, with score 2 min². Changing both settings to $(1.1,-0.1)$ preserves calibration and lowers the score to 1.46 min². Changing only the rate to $(1,-0.1)$ gives score 1.54 min², but its calibration residual is $-0.1$. That move is rejected.

Both arrows reduce the score. Only the teal arrow stays on the calibration line. Feasibility determines which improvement is usable.
<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-2_constrained_optimization_lagrange_multipliers/assets/feasible_clock_changes.png" alt="From settings 1 and 0 with score 2, changing to 1.1 and minus 0.1 preserves calibration and gives score 1.46. Changing only the rate to minus 0.1 gives score 1.54 but breaks calibration and is rejected." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The code computes each score from the four errors and checks calibration before accepting a comparison.


In [ ]:
import numpy as np

OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])  # h
CALIBRATION_TIME = 1.0  # h
REQUIRED_CORRECTION = 1.0  # min
CHECK_TOLERANCE = 1e-9
CONSTRAINT_GRADIENT = np.array([1.0, 1.0])
TANGENT_DIRECTION = np.array([1.0, -1.0])


def evaluate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    errors = OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES
    gradient = 2 * np.array([errors.sum(), OBSERVATION_TIMES @ errors])
    residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    return {"decision": np.array([correction, rate]), "response": errors,
            "objective": float(errors @ errors), "gradient": gradient,
            "feasible": bool(abs(residual) <= CHECK_TOLERANCE),
            "directional_derivative": float(gradient @ TANGENT_DIRECTION)}


In [ ]:
for decision in ([1.0, 0.0], [1.1, -0.1], [1.0, -0.1]):
    result = evaluate_clock(decision)
    status = "feasible" if result["feasible"] else "REJECT: calibration fails"
    print(f"Settings {decision}: score {result['objective']:.2f} min² | {status}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Read a zero slope along the allowed line

The **gradient** collects the two objective slopes, one for each setting. The slope along $(1,-1)$ is the $q$ slope minus the $r$ slope. This is the feasible-direction slope, also called a directional derivative.

At $(1,0)$, the slopes are $(0,6)$, so the feasible-direction slope is $0-6=-6$. Increasing $q$ while decreasing $r$ improves the score locally. At $(1.5,-0.5)$, the slopes are $(-2,-2)$, so the same calculation gives zero. The objective still has nonzero individual slopes, but they cancel along the allowed line.

The equality multiplier $\nu=2$ balances those individual slopes in the Lagrangian: $-2+2=0$ for each setting. The code displays the feasible-direction slope and this balance.


In [ ]:
for decision in ([1.0, 0.0], [1.5, -0.5]):
    result = evaluate_clock(decision)
    print(f"Settings {decision}: objective slopes {result['gradient']}, "
          f"feasible-direction slope {result['directional_derivative']:.1f}")
optimum = evaluate_clock([1.5, -0.5])
print("Lagrangian slopes:", optimum["gradient"] + 2 * CONSTRAINT_GRADIENT)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Confirm what nearby allowed moves do

At the candidate $(1.5,-0.5)$, a calibrated change has settings $(1.5+s,-0.5-s)$, where $s$ is a numerical step. The score is

$$
\phi(1.5+s,-0.5-s)=0.5+6s^2.
$$

For $s=-0.1$ or $s=0.1$, the score is 0.56 min². For $s=0$, it is 0.50 min². The nonnegative square establishes the minimum for every feasible step; the printed values make that shape concrete.


In [ ]:
for step in (-0.1, 0.0, 0.1):
    result = evaluate_clock([1.5 + step, -0.5 - step])
    print(f"Step {step:+.1f}: feasible = {result['feasible']}, "
          f"score = {result['objective']:.2f} min²")
